# 04 — Homography Pipeline Evaluation (11 models)

Pipeline per gambar per model:
```
Image → Model.predict() → KeypointResult
      → Merge dengan canonical pitch coords (kpt_id matching)
      → cv2.findHomography(RANSAC, threshold=5.0px)
      → Validity check (condition number < 1000)
      → Log: is_four_points, is_H_valid, n_inliers, n_outliers
```

**Output:** `artifacts/logs/evaluation/homography_results.csv`

**Prereq:** Weights semua model sudah tersedia (dari notebook 02a–02d).

In [ ]:
# ── Cell 1: Install + Mount ───────────────────────────────────────────────────
!pip install ultralytics openmim pyyaml tqdm opencv-python-headless --quiet
!mim install mmengine mmcv mmpose --quiet

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Environment & Path Setup — kompatibel Kaggle DAN Colab (auto-detect)
# ══════════════════════════════════════════════════════════════════════
import os, sys

def _detect_env():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

ENV = _detect_env()
print(f'Environment terdeteksi: {ENV}')

# ⚠️ GANTI dengan URL repo GitHub kamu sendiri (isi SEKALI saja per notebook)
GITHUB_REPO = 'https://github.com/atangorp/soccer-homography-benchmark.git'

def _link(src, dst):
    """Symlink src -> dst kalau src ada dan dst belum ada. Aman dipanggil berkali-kali."""
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

if ENV == 'kaggle':
    PROJECT_ROOT = '/kaggle/working/soccer-homography-benchmark'
    if not os.path.exists(PROJECT_ROOT):
        os.system(f'git clone -q {GITHUB_REPO} "{PROJECT_ROOT}"')
    sys.path.insert(0, PROJECT_ROOT)

    # WAJIB: attach dataset "soccer-field-raw" via "+ Add Input" di sidebar kanan
    # (ganti 'soccer-field-raw' kalau slug dataset kamu berbeda)
    _link('/kaggle/input/soccer-field-raw',
          f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8')
    # Weights hasil training — attach masing-masing notebook output via "+ Add Input"
    _link('/kaggle/input/02a-train-yolo11/soccer-homography-benchmark/artifacts/weights/yolo11',
          f'{PROJECT_ROOT}/artifacts/weights/yolo11')
    _link('/kaggle/input/02b-train-hrnet/soccer-homography-benchmark/artifacts/weights/hrnet',
          f'{PROJECT_ROOT}/artifacts/weights/hrnet')
    _link('/kaggle/input/02c-train-vitpose/soccer-homography-benchmark/artifacts/weights/vitpose',
          f'{PROJECT_ROOT}/artifacts/weights/vitpose')
    _link('/kaggle/input/02d-train-detr/soccer-homography-benchmark/artifacts/weights/detr',
          f'{PROJECT_ROOT}/artifacts/weights/detr')
    # Gambar test set (dibutuhkan evaluasi visual + GT label)
    for _split in ['train', 'val', 'test']:
        _link(f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8/{_split}/images',
              f'{PROJECT_ROOT}/data/converted/images/{_split}')

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/soccer-homography-benchmark'
    sys.path.insert(0, PROJECT_ROOT)

else:
    PROJECT_ROOT = os.getcwd()
    sys.path.insert(0, PROJECT_ROOT)

import yaml, json, pandas as pd, numpy as np, torch
from tqdm import tqdm

with open(f'{PROJECT_ROOT}/configs/eval_global.yaml') as f:
    CFG = yaml.safe_load(f)

TEST_IMAGES = f'{PROJECT_ROOT}/data/converted/images/test'
OUTPUT_DIR  = f'{PROJECT_ROOT}/artifacts/logs/evaluation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'RANSAC threshold : {CFG["ransac_threshold"]} px')
print(f'Min keypoints    : {CFG["min_keypoints_for_homography"]}')
print(f'Max cond number  : {CFG["max_condition_number"]}')
print(f'PROJECT_ROOT : {PROJECT_ROOT}')


In [ ]:
# ── Cell 3: Import utilities ──────────────────────────────────────────────────
from src.geometry.pitch_reference import get_dst_dataframe
from src.geometry.homography      import compute_homography_from_df
from src.geometry.metrics         import classify_image_source
from src.models.yolo11_wrapper    import YOLO11Wrapper
from src.models.hrnet_wrapper     import HRNetWrapper
from src.models.vitpose_wrapper   import ViTPoseWrapper
from src.models.detr_wrapper      import DeformableDETRWrapper

# Canonical pitch keypoints — single source of truth
DF_DST = get_dst_dataframe()
print(f'Pitch reference loaded: {len(DF_DST)} keypoints')
print(DF_DST.head(5))

In [ ]:
# ── Cell 4: Model registry (sama dengan notebook 03) ─────────────────────────
W = f'{PROJECT_ROOT}/artifacts/weights'

MODEL_REGISTRY = [
    {'family': 'yolo11', 'variant': 'small',  'type': 'yolo11',
     'weights': f'{W}/yolo11/small/run/weights/best.pt'},
    {'family': 'yolo11', 'variant': 'medium', 'type': 'yolo11',
     'weights': f'{W}/yolo11/medium/run/weights/best.pt'},
    {'family': 'yolo11', 'variant': 'xlarge', 'type': 'yolo11',
     'weights': f'{W}/yolo11/xlarge/run/weights/best.pt'},
    {'family': 'hrnet', 'variant': 'w18', 'type': 'hrnet',
     'weights': f'{W}/hrnet/w18/best_coco_AP.pth',
     'config':  f'{W}/hrnet/w18/hrnet_w18_config.py'},
    {'family': 'hrnet', 'variant': 'w32', 'type': 'hrnet',
     'weights': f'{W}/hrnet/w32/best_coco_AP.pth',
     'config':  f'{W}/hrnet/w32/hrnet_w32_config.py'},
    {'family': 'hrnet', 'variant': 'w48', 'type': 'hrnet',
     'weights': f'{W}/hrnet/w48/best_coco_AP.pth',
     'config':  f'{W}/hrnet/w48/hrnet_w48_config.py'},
    {'family': 'vitpose', 'variant': 'small', 'type': 'vitpose',
     'weights': f'{W}/vitpose/small/best_coco_AP.pth',
     'config':  f'{W}/vitpose/small/vitpose_small_config.py'},
    {'family': 'vitpose', 'variant': 'base',  'type': 'vitpose',
     'weights': f'{W}/vitpose/base/best_coco_AP.pth',
     'config':  f'{W}/vitpose/base/vitpose_base_config.py'},
    {'family': 'vitpose', 'variant': 'large', 'type': 'vitpose',
     'weights': f'{W}/vitpose/large/best_coco_AP.pth',
     'config':  f'{W}/vitpose/large/vitpose_large_config.py'},
    {'family': 'detr', 'variant': 'r50',  'type': 'detr',
     'weights': f'{W}/detr/r50/checkpoint_best.pth'},
    {'family': 'detr', 'variant': 'r101', 'type': 'detr',
     'weights': f'{W}/detr/r101/checkpoint_best.pth'},
]

def build_model(spec):
    t, w, v = spec['type'], spec['weights'], spec['variant']
    c = CFG['conf_threshold']
    if t == 'yolo11':  return YOLO11Wrapper(w, conf_threshold=c, variant=v)
    if t == 'hrnet':   return HRNetWrapper(w, spec['config'], conf_threshold=c, variant=v)
    if t == 'vitpose': return ViTPoseWrapper(w, spec['config'], conf_threshold=c, variant=v)
    if t == 'detr':    return DeformableDETRWrapper(w, conf_threshold=c, variant=v)

In [ ]:
# ── Cell 5: Homography pipeline per model ────────────────────────────────────
import gc
import glob

img_paths = sorted(
    glob.glob(f'{TEST_IMAGES}/*.jpg') +
    glob.glob(f'{TEST_IMAGES}/*.png')
)
print(f'Test images: {len(img_paths)}')

all_rows = []

for spec in MODEL_REGISTRY:
    model_name = f"{spec['family']}-{spec['variant']}"

    if not os.path.exists(spec['weights']):
        print(f'\n⏭️  Skip {model_name} — weights belum ada')
        continue

    print(f'\n▶️  {model_name}')
    model = build_model(spec)
    model.load_model()

    for img_path in tqdm(img_paths, desc=model_name):
        img_name     = os.path.basename(img_path)
        image_source = classify_image_source(img_name)

        # Inference
        results = model.predict(img_path)

        if not results:
            all_rows.append({
                'image_name': img_name, 'image_source': image_source,
                'model_family': spec['family'], 'model_variant': spec['variant'],
                'instance_detected': 0, 'is_four_points': False,
                'is_H_valid': False, 'condition_number': np.inf,
                'n_inliers': 0, 'n_outliers': 0, 'n_correspondences': 0,
            })
            continue

        # Build df_src dari top-1 detection
        top1 = results[0]
        df_src_rows = top1.to_dataframe_rows()
        if not df_src_rows:
            continue

        import pandas as pd
        df_src = pd.DataFrame(df_src_rows)

        # Hitung homography
        homo = compute_homography_from_df(
            df_src=df_src,
            df_dst=DF_DST,
            ransac_threshold=CFG['ransac_threshold'],
            min_keypoints=CFG['min_keypoints_for_homography'],
        )

        all_rows.append({
            'image_name':        img_name,
            'image_source':      image_source,
            'model_family':      spec['family'],
            'model_variant':     spec['variant'],
            'instance_detected': 1,
            'is_four_points':    homo['is_four_points'],
            'is_H_valid':        homo['is_H_valid'],
            'condition_number':  homo['condition_number'],
            'n_inliers':         homo['n_inliers'],
            'n_outliers':        homo['n_outliers'],
            'n_correspondences': homo['n_correspondences'],
        })

    # Unload
    model.unload()
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print(f'   🧹 Memory cleared')

print(f'\n✅ Selesai: {len(all_rows)} rows')

In [ ]:
# ── Cell 6: Save + quick summary ─────────────────────────────────────────────
df_homo = pd.DataFrame(all_rows)
out_path = f'{OUTPUT_DIR}/homography_results.csv'
df_homo.to_csv(out_path, index=False)
print(f'💾 Saved: {out_path}')

# Summary per model
summary = df_homo.groupby(['model_family', 'model_variant']).agg(
    detection_rate    = ('instance_detected', 'mean'),
    four_point_rate   = ('is_four_points', 'mean'),
    validity_rate     = ('is_H_valid', 'mean'),
    mean_inliers      = ('n_inliers', 'mean'),
    mean_cond_number  = ('condition_number', lambda x: x[x < np.inf].mean()),
    n_images          = ('image_name', 'count'),
).reset_index().round(3)

print('\n📊 Homography validity rate per model:')
display(summary.sort_values('validity_rate', ascending=False))

In [ ]:
# ── Cell 7: Breakdown per image source ───────────────────────────────────────
# Analisis broadcast vs scouting — ini adalah finding kunci untuk paper

from src.geometry.metrics import homography_validity_rate

for source in df_homo['image_source'].unique():
    subset = df_homo[df_homo['image_source'] == source]
    print(f'\n── Image source: {source} (n={len(subset)}) ──')
    src_summary = subset.groupby(['model_family', 'model_variant'])['is_H_valid'].mean()
    print(src_summary.sort_values(ascending=False).to_string())